# Mobile Addiction Machine Learning Model & API
This notebook generates synthetic data, trains a model, and serves a FastAPI application.

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
import pickle

# 1. Generate Synthetic Data
np.random.seed(42)
n_samples = 1000

data = {
    'screen_time': np.random.uniform(1, 15, n_samples),
    'social_media': np.random.uniform(0, 8, n_samples),
    'phone_checks': np.random.randint(10, 300, n_samples),
    'usage_before_sleep': np.random.uniform(0, 4, n_samples),
    'usage_after_waking': np.random.randint(0, 60, n_samples),
    'gaming_time': np.random.uniform(0, 10, n_samples),
    'interruption': np.random.choice([0, 1], n_samples, p=[0.6, 0.4])
}

df = pd.DataFrame(data)

# Addiction labeling logic
# Calculate a raw score to determine labels
raw_score = (df['screen_time'] * 2 + df['social_media'] * 3 + 
             (df['phone_checks'] / 20) + df['usage_before_sleep'] * 4 + 
             (df['usage_after_waking'] / 10) + df['gaming_time'] * 1.5 + 
             (df['interruption'] * 15))

def map_addiction_level(score):
    if score < 35:
        return 0 # Low
    elif score < 60:
        return 1 # Moderate
    else:
        return 2 # High

df['target'] = raw_score.apply(map_addiction_level)
print("Distribution of Addiction Levels:")
print(df['target'].value_counts())

# 2. Train Model
X = df.drop('target', axis=1)
y = df['target']
model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X, y)
print("Model trained.")

# 3. Save Model
with open('model.pkl', 'wb') as f:
    pickle.dump(model, f)
print("Model saved to model.pkl")

In [ ]:
import nest_asyncio
import uvicorn
from fastapi import FastAPI
from pydantic import BaseModel
import pickle
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# Load model
with open('model.pkl', 'rb') as f:
    model = pickle.load(f)

app = FastAPI()

class UserInput(BaseModel):
    screen_time: float
    social_media: float
    phone_checks: int
    usage_before_sleep: float
    usage_after_waking: int
    gaming_time: float
    interruption: int

@app.post("/predict")
def predict_addiction(data: UserInput):
    features = np.array([[
        data.screen_time,
        data.social_media,
        data.phone_checks,
        data.usage_before_sleep,
        data.usage_after_waking,
        data.gaming_time,
        data.interruption
    ]])
    prediction = model.predict(features)[0]
    level = int(prediction)
    
    levels_map = {0: "Low Addiction", 1: "Moderate Addiction", 2: "High Addiction"}
    
    return {
        "prediction": levels_map.get(level, "Unknown"),
        "level_index": level
    }

print("Starting FastAPI server...")
nest_asyncio.apply()
uvicorn.run(app, host="127.0.0.1", port=5000)
